# =======
# Initializion
# =======

In [5]:
import pandas as pd
import numpy as np

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from statsbombpy import sb
import meteostat as ms
from datetime import datetime, timedelta
import helper_functions as f

# spark biblioteker fra labene
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType
)

In [6]:
# Create the SparkSession - the entry point for all Spark functionality
spark = SparkSession.builder \
    .appName("DAT535-Project-FootballStats") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/10 14:38:07 WARN Utils: Your hostname, LaptopEHM, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/10 14:38:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 14:38:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.2.1
Application ID: local-1794317890002
Spark UI: http://10.255.255.254:4040


# ========
# Bronze layer
# ========

### OpenData Football data

In [ ]:
competitions = sb.competitions()

la_liga_seasons = competitions[competitions["competition_id"] == 11]["season_id"]
la_liga_seasons = la_liga_seasons.to_numpy()

frames = []
for season_id in la_liga_seasons:
    frames.append(sb.matches(11, season_id))
matches_df = pd.concat(frames, ignore_index= True)


sb.competition_events()
# match_ids = matches_df["match_id"].to_numpy()
# frames = []

# for match_id in match_ids:
#     frames.append(sb.events(match_id))
# events_df = pd.concat(frames, ignore_index=True)

In [ ]:
z = sb.competition_events(country="Spain", division="La Liga", season="2016/2017")
z.to_csv("_test.csv")

### Meteostat Weather data

In [ ]:
matches_df["stadium"].nunique()

In [ ]:
locations = matches_df[["stadium", "match_date", "kick_off"]]

for i, row in locations.iterrows():
    lat, long = f.get_coordinates(row["stadium"])

    # Creates a datetime from a string (e.g. datetime.striptime("2026-11-03 17:07:01.000", "%Y-%m-%d %H:%M:%S.%f"))
    match_start = datetime.strptime(f"{row["match_date"]} {row["kick_off"]}", "%Y-%m-%d %H:%M:%S.%f") \
        .replace(minute=0, second=0, microsecond=0)
    POINT = ms.Point(lat, long)
    START = match_start - timedelta(hours=3)
    END = match_start + timedelta(hours=3)

    stations = ms.stations.nearby(POINT, limit=4)

    time_series = ms.hourly(stations, START, END)
    weather = ms.interpolate(time_series, POINT).fetch()

# =======
# Silver layer
# =======

# =======
# Gold layer
# =======